## 02-seq2seq: Experiment 노트북

### 목표
- LSTM으로 구성한 Seq-to-Seq 모델을 작은 batch에서 over-fitting
- Synthetic Reverse Task에서 학습 확인
- Attention을 이용한 Seq-to-seq에서 Attention Map 시각화
- 번역 태스크에서 Attention 유무에 따른 성능 비교

### 1. Import 및 설정

In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
import math
import copy
import random
from typing import Tuple, Dict, Optional

import torch
import torch.nn as nn
from torch import Tensor
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from model import Seq2Seq
from data import synthetic_reverse_dataset

In [3]:
def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.random.manual_seed(seed)

set_seed()

### 2. 작은 Batch에서 Overfitting 확인

In [7]:
batch_size = 16
min_seq, max_seq = 3, 6
embedding_size = 32
hidden_size = 64
vocab_size = 13

PAD_TOKEN = 0
BOS_TOKEN = 1
EOS_TOKEN = 2

source, target_input, target_output, valid_mask = \
    synthetic_reverse_dataset(batch_size=batch_size, min_seq=min_seq, max_seq=max_seq)

model = Seq2Seq(vocab_size=vocab_size, embedding_size=embedding_size, hidden_size=hidden_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)
eval_mask = target_output != PAD_TOKEN

In [ ]:
loss_history, acc_history = [], []
for i in range(100):
    optimizer.zero_grad()
    logits = model(source, target_input, valid_mask)
    loss = loss_fn(logits.reshape(-1, vocab_size), target_output.reshape(-1))
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())
    correct = (logits.argmax(dim=-1) == target_output) & eval_mask
    acc_history.append((torch.sum(correct) / torch.sum(eval_mask)).item())

    if i % 10 == 0:
        print(f"Step {i}: loss={loss.item():.2f}, acc={(torch.sum(correct) / torch.sum(eval_mask)):.2f}")
    

Step 0: loss=2.57, acc=0.07
Step 10: loss=1.09, acc=0.61
Step 20: loss=0.23, acc=0.98
Step 30: loss=0.04, acc=1.00
Step 40: loss=0.01, acc=1.00
Step 50: loss=0.01, acc=1.00
Step 60: loss=0.00, acc=1.00
Step 70: loss=0.00, acc=1.00
Step 80: loss=0.00, acc=1.00
Step 90: loss=0.00, acc=1.00
